# Inverted Index

The text files are placed in some folder.

The **wholeTextFiles** will read the files, and output a tuple (filename, fulltext)

wholeTextFiles -> (filename, fulltext)

In the first step (flatMap), we map our existing key/value pair (filename, fulltext) to a new key-value pair comprising of document-id (i,e, filename) and token (each word in the fulltext) as the key and 1 (representing count) as the value. We will be using the flatMap transformation here to combine all the tokens in a single list.

 flatMap -> ((filename, token), 1)

In the second step (reduceByKey), we will be grouping the key/value pairs with the common key and further aggregate the values for the same key to get the term frequency for a particular word corresponding to its filename.

reduceByKey -> ((filename, token), 1+1+1...)

In the third step, we will change the key/value pairs to a new set of key/ value/pairs with tokens as the key and its filename and respective term frequency as the values.

map -> (token, (filename, count)), notice the count here is also called "Term Frequency"

In [1]:
from pyspark import SparkConf, SparkContext
import os, re

sc = SparkContext.getOrCreate(SparkConf())

#for processing larger dataset
sc._conf.set('spark.executor.memory','8g')
sc._conf.set('spark.driver.memory','8g')
sc._conf.set('spark.driver.maxResultsSize','0')


In [2]:
                #.flatMap(lambda x: [((os.path.basename(x[0]).split(".")[0] ,i) ,1) for i in re.split('\\W', x[1])])\
inverted_index = sc.wholeTextFiles('data/books_100M')\
                .flatMap(lambda x: [((os.path.basename(x[0]),i.lower()), 1) for i in re.split('\\W', x[1])])\
                .reduceByKey(lambda a, b: a + b)\
                .map(lambda x: (x[0][1],(x[0][0],x[1])))\
                .filter(lambda x: x[0] !='')

output = inverted_index.take(10)
for i in range(10):
    print(output[i])

#output.saveAsTextFile('invertedIndex')

('for', ('1.txt', 223))
('states', ('1.txt', 144))
('exact', ('1.txt', 3))
('annual', ('1.txt', 2))
('every', ('1.txt', 26))
('caps', ('1.txt', 1))
('endowed', ('1.txt', 2))
('having', ('1.txt', 14))
('endeavoured', ('1.txt', 4))
('boundaries', ('1.txt', 2))


In [3]:
inverted_index = inverted_index.reduceByKey(lambda a, b: a + b).sortByKey()
inverted_index.take(10)

[('0',
  ('105.txt',
   1,
   '109.txt',
   1,
   '156.txt',
   1,
   '119.txt',
   2,
   '155.txt',
   2,
   '145.txt',
   2,
   '108.txt',
   3,
   '176.txt',
   2,
   '141.txt',
   2,
   '102.txt',
   1,
   '143.txt',
   2,
   '177.txt',
   2,
   '12.txt',
   2,
   '125.txt',
   2,
   '101.txt',
   6,
   '117.txt',
   1,
   '11.txt',
   3,
   '118.txt',
   12,
   '14.txt',
   1083,
   '16.txt',
   2,
   '91.txt',
   2,
   '94.txt',
   2,
   '29.txt',
   235,
   '37.txt',
   241,
   '79.txt',
   13,
   '80.txt',
   51,
   '93.txt',
   2,
   '32.txt',
   2,
   '67.txt',
   1,
   '43.txt',
   2,
   '84.txt',
   2,
   '25.txt',
   1087,
   '41.txt',
   2,
   '60.txt',
   2,
   '86.txt',
   3,
   '34.txt',
   13,
   '87.txt',
   1496,
   '202.txt',
   3,
   '48.txt',
   1089,
   '106.txt',
   1,
   '134.txt',
   2,
   '104.txt',
   1,
   '171.txt',
   2,
   '160.txt',
   2,
   '133.txt',
   2,
   '153.txt',
   2,
   '115.txt',
   55933,
   '121.txt',
   2,
   '112.txt',
   1,
   '116.txt

In [4]:
#save inverted index to a folder
# make sure there is no the following folder yet, otherwise, you need delete it first
# save the index in the same folder/directory of this code
inverted_index.saveAsTextFile('./InvIndex_100m')

In [6]:
#wholeTextFiles -> (filename, fulltext)
rdd1=sc.wholeTextFiles('data/books_100M')

#flatMap -> ((filename, token), 1)
#rdd2 = rdd1.flatMap(lambda x: [((i，os.path.basename(x[0]).split(".")[0]) ,1) for i in re.split('\\W', x[1])])
rdd2 = rdd1.flatMap(lambda x: [((i.lower(),os.path.basename(x[0])), 1) for i in re.split('\\W', x[1])])
    
output = rdd2.take(10)
for i in range(10):
    print(output[i])



(('', '1.txt'), 1)
(('', '1.txt'), 1)
(('', '1.txt'), 1)
(('', '1.txt'), 1)
(('', '1.txt'), 1)
(('', '1.txt'), 1)
(('', '1.txt'), 1)
(('', '1.txt'), 1)
(('', '1.txt'), 1)
(('', '1.txt'), 1)


In [7]:
#reduceByKey -> ((filename, token), 1+1+1...)
rdd3 = rdd2.reduceByKey(lambda a, b: a + b)
rdd3.take(10)

[(('made', '1.txt'), 22),
 (('across', '1.txt'), 2),
 (('their', '1.txt'), 81),
 (('deriving', '1.txt'), 2),
 (('whenever', '1.txt'), 5),
 (('guards', '1.txt'), 2),
 (('naturalization', '1.txt'), 3),
 (('conjured', '1.txt'), 2),
 (('levy', '1.txt'), 2),
 (('chains', '1.txt'), 4)]

In [8]:
#map -> (token, (filename, count))
inverted_index = rdd3.map(lambda x: (x[0][0],(x[0][1],x[1]))).filter(lambda x: x[0] !='')
inverted_index.take(10)

[('made', ('1.txt', 22)),
 ('across', ('1.txt', 2)),
 ('their', ('1.txt', 81)),
 ('deriving', ('1.txt', 2)),
 ('whenever', ('1.txt', 5)),
 ('guards', ('1.txt', 2)),
 ('naturalization', ('1.txt', 3)),
 ('conjured', ('1.txt', 2)),
 ('levy', ('1.txt', 2)),
 ('chains', ('1.txt', 4))]

In [9]:
#reduceByKey -> (token, (filename1, count1, filename2, count2, ...))
inverted_index = inverted_index.reduceByKey(lambda a, b: a + b).sortByKey()
inverted_index.take(5)

[('0',
  ('102.txt',
   1,
   '109.txt',
   1,
   '118.txt',
   12,
   '119.txt',
   2,
   '143.txt',
   2,
   '177.txt',
   2,
   '156.txt',
   1,
   '101.txt',
   6,
   '11.txt',
   3,
   '14.txt',
   1083,
   '108.txt',
   3,
   '145.txt',
   2,
   '117.txt',
   1,
   '141.txt',
   2,
   '12.txt',
   2,
   '16.txt',
   2,
   '105.txt',
   1,
   '155.txt',
   2,
   '125.txt',
   2,
   '176.txt',
   2,
   '87.txt',
   1496,
   '37.txt',
   241,
   '67.txt',
   1,
   '86.txt',
   3,
   '93.txt',
   2,
   '29.txt',
   235,
   '84.txt',
   2,
   '32.txt',
   2,
   '202.txt',
   3,
   '25.txt',
   1087,
   '60.txt',
   2,
   '41.txt',
   2,
   '43.txt',
   2,
   '48.txt',
   1089,
   '34.txt',
   13,
   '91.txt',
   2,
   '94.txt',
   2,
   '79.txt',
   13,
   '80.txt',
   51,
   '122.txt',
   2,
   '111.txt',
   2,
   '149.txt',
   2,
   '134.txt',
   2,
   '115.txt',
   55933,
   '165.txt',
   2,
   '113.txt',
   2,
   '121.txt',
   2,
   '152.txt',
   1,
   '174.txt',
   2,
   '15.txt'

In [10]:
sc.stop()